In [1]:
%%capture
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth
else:
    import torch
    v = re.match(r'[\d]{1,}\.[\d]{1,}', str(torch.__version__)).group(0)
    xformers = 'xformers==' + {'2.10':'0.0.34','2.9':'0.0.33.post1','2.8':'0.0.32.post2'}.get(v, "0.0.34")

    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth_zoo bitsandbytes accelerate {xformers} peft trl triton unsloth

!pip install transformers==4.56.2
!pip install --no-deps trl==0.22.2

In [2]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 4096
load_in_4bit = True

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Phi-4-unsloth-bnb-4bit",
    max_seq_length=max_seq_length,
    load_in_4bit=load_in_4bit,
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.3.4: Fast Llama patching. Transformers: 4.56.2.
   \\   /|    NVIDIA A100-SXM4-80GB. Num GPUs = 1. Max memory: 79.251 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 8.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.34. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

model-00001-of-00003.safetensors:   0%|          | 0.00/4.97G [00:00<?, ?B/s]

model-00002-of-00003.safetensors:   0%|          | 0.00/4.39G [00:00<?, ?B/s]

model-00003-of-00003.safetensors:   0%|          | 0.00/1.03G [00:00<?, ?B/s]

...

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [3]:
import re
import torch
from unsloth import FastLanguageModel
from datasets import load_dataset
from google.colab import userdata

# 1. Retrieve your secure token
hf_token = userdata.get('HF_TOKEN')

# 2. Load the Fine-Tuned Model (This automatically loads the base model + your adapters)
print("Loading fine-tuned model from Hugging Face...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "Jaiccc/model_0_streaming_timestamp", # Your private repo
    max_seq_length = 4096,
    load_in_4bit = True,
    token = hf_token,
)

# Enable native 2x faster inference
FastLanguageModel.for_inference(model)

# 3. Load the dataset
print("Loading dataset...")
dataset = load_dataset("Jaiccc/model0_boundary_predict_streaming", split="train")

# 4. Initialize counters for both models
total_samples = 200
ft_correct_count = 0
base_correct_count = 0

print(f"\nScanning {total_samples} samples side-by-side...\n")

for i in range(200, total_samples + 200):
    sample = dataset[i]

    # Extract components
    instruction = sample['instruction']
    input_data = sample['input']
    expected_output = sample['output'].strip()

    # Format the prompt using ChatML structure
    combined_user_text = f"{instruction}\n\n{input_data}"
    prompt = f"<|im_start|>user<|im_sep|>{combined_user_text}<|im_end|><|im_start|>assistant<|im_sep|>"
    inputs = tokenizer(prompt, return_tensors="pt").input_ids.to("cuda")

    # ==========================================
    # GENERATE 1: FINE-TUNED MODEL PREDICTION
    # ==========================================
    outputs_ft = model.generate(
        input_ids=inputs,
        max_new_tokens=64,
        use_cache=True,
        temperature=0.1,
    )
    raw_output_ft = tokenizer.batch_decode(outputs_ft, clean_up_tokenization_spaces=True)[0]

    m_ft = re.search(r'<\|im_start\|>assistant<\|im_sep\|>(.*?)<\|im_end\|>', raw_output_ft, re.S)
    ft_result = m_ft.group(1).strip() if m_ft else raw_output_ft.split("assistant")[-1].strip()

    # ==========================================
    # GENERATE 2: BASE MODEL PREDICTION
    # ==========================================
    # This safely turns off your LoRA weights temporarily!
    with model.disable_adapter():
        outputs_base = model.generate(
            input_ids=inputs,
            max_new_tokens=64,
            use_cache=True,
            temperature=0.1,
        )
    raw_output_base = tokenizer.batch_decode(outputs_base, clean_up_tokenization_spaces=True)[0]

    m_base = re.search(r'<\|im_start\|>assistant<\|im_sep\|>(.*?)<\|im_end\|>', raw_output_base, re.S)
    base_result = m_base.group(1).strip() if m_base else raw_output_base.split("assistant")[-1].strip()

    # ==========================================
    # COMPARE AND PRINT
    # ==========================================
    is_ft_correct = "✅" if ft_result == expected_output else "❌"
    is_base_correct = "✅" if base_result == expected_output else "❌"

    if ft_result == expected_output:
        ft_correct_count += 1
    if base_result == expected_output:
        base_correct_count += 1

    print(f"--- Sample {i} ---")
    print(f"EXPECTED:   {expected_output}")
    print(f"BASE MODEL: {base_result} {is_base_correct}")
    print(f"FINE-TUNED: {ft_result} {is_ft_correct}")
    print("-" * 30)

# 5. Final Summary Comparison
ft_accuracy = (ft_correct_count / total_samples) * 100
base_accuracy = (base_correct_count / total_samples) * 100

print("\n" + "="*30)
print("🏆 FINAL ACCURACY COMPARISON 🏆")
print("="*30)
print(f"Base Model: {base_accuracy:.2f}% ({base_correct_count}/{total_samples})")
print(f"Fine-Tuned: {ft_accuracy:.2f}% ({ft_correct_count}/{total_samples})")
print(f"Improvement: +{ft_accuracy - base_accuracy:.2f}%")

Loading fine-tuned model from Hugging Face...
==((====))==  Unsloth 2026.3.4: Fast Llama patching. Transformers: 4.56.2.
   \\   /|    NVIDIA A100-SXM4-80GB. Num GPUs = 1. Max memory: 79.251 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 8.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.34. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

model-00001-of-00003.safetensors:   0%|          | 0.00/4.97G [00:00<?, ?B/s]

model-00002-of-00003.safetensors:   0%|          | 0.00/4.39G [00:00<?, ?B/s]

model-00003-of-00003.safetensors:   0%|          | 0.00/1.03G [00:00<?, ?B/s]

...

Loading dataset...


train.jsonl:   0%|          | 0.00/4.69M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/1374 [00:00<?, ? examples/s]

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.



Scanning 200 samples side-by-side...

--- Sample 200 ---
EXPECTED:   35.290792, new event
BASE MODEL: 35.290792, old event ❌
FINE-TUNED: 35.290792, new event ✅
------------------------------
--- Sample 201 ---
EXPECTED:   47.225664, new event
BASE MODEL: 47.225664, new event ✅
FINE-TUNED: 47.225664, new event ✅
------------------------------
--- Sample 202 ---
EXPECTED:   49.904836, old event
BASE MODEL: 49.904836, old event ✅
FINE-TUNED: 49.904836, old event ✅
------------------------------
--- Sample 203 ---
EXPECTED:   63.595226, new event
BASE MODEL: 63.595226, new event ✅
FINE-TUNED: 63.595226, new event ✅
------------------------------
--- Sample 204 ---
EXPECTED:   69.555972, new event
BASE MODEL: 69.555972, new event ✅
FINE-TUNED: 69.555972, new event ✅
------------------------------
--- Sample 205 ---
EXPECTED:   72.743598, old event
BASE MODEL: 72.743598, old event ✅
FINE-TUNED: 72.743598, old event ✅
------------------------------
--- Sample 206 ---
EXPECTED:   117.528187, 